<a href="https://colab.research.google.com/github/cesure10/HTML_Practice/blob/main/deep_learing_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint

import tensorflow as tf
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Load dataset
df = pd.read_csv("Blood glucose readings.csv")

# Strip extra spaces in column names
df.columns = df.columns.str.strip()

In [ ]:
# Rename columns properly
df.columns = [
    "Age",
    "Blood Glucose Level (BGL)",
    "Diastolic BP",
    "Systolic BP",
    "Heart Rate",
    "Body Temperature",
    "SPO2",
    "Sweating (Y/N)",
    "Shivering (Y/N)",
    "Diabetic (D/N)"
]

# Remove rows with NaN (if any)
df = df.dropna()

# Convert categorical (Y/N, D/N) to numeric
df = df.replace({'Y': 1, 'N': 0, 'D': 1, 'NonDiabetic': 0, 'n': 0, 'y': 1, 'N': 0})


In [ ]:
# Drop the extra header row (the one containing 'Age' string instead of numeric)
df = df[df["Age"] != "Age"]

# Convert all numeric columns to float
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop NaN values after conversion (if any)
df = df.dropna().reset_index(drop=True)

print("Final Cleaned Dataset Preview:")
display(df.head())
print("\n Data types:\n", df.dtypes)
print("\n Data Shape:", df.shape)


In [ ]:
# Define target and features
target = "Blood Glucose Level (BGL)"
features = [col for col in df.columns if col not in [target]]

# Scale the data for LSTM
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[[target] + features])

# Convert back to DataFrame for clarity
scaled_df = pd.DataFrame(scaled_data, columns=[target] + features)

# Define sequence length
TIME_STEPS = 10  # use past 10 readings to predict the next

# Create sequences
def create_sequences(data, time_steps=TIME_STEPS):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data.iloc[i:(i + time_steps)].values)
        y.append(data.iloc[i + time_steps][target])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_df)

print(f" LSTM Input Shape: {X.shape}")
print(f" LSTM Output Shape: {y.shape}")


In [ ]:
split_index = int(0.8 * len(X))
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

In [ ]:
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1)  # output layer for regression (glucose level)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
y_pred = model.predict(X_test)

# Inverse scaling only for target column
inv_y_test = scaler.inverse_transform(
    np.concatenate([y_test.reshape(-1,1), np.zeros((len(y_test), len(features)))], axis=1)
)[:,0]

inv_y_pred = scaler.inverse_transform(
    np.concatenate([y_pred, np.zeros((len(y_pred), len(features)))], axis=1)
)[:,0]

mae = mean_absolute_error(inv_y_test, inv_y_pred)
rmse = np.sqrt(mean_squared_error(inv_y_test, inv_y_pred))
r2 = r2_score(inv_y_test, inv_y_pred)

print(f" Mean Absolute Error (MAE): {mae:.3f}")
print(f" Root Mean Squared Error (RMSE): {rmse:.3f}")
print(f" R² Score: {r2:.3f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(6,4))
plt.plot(inv_y_test[:200], label='Actual Glucose', linewidth=2)
plt.plot(inv_y_pred[:200], label='Predicted Glucose', linestyle='--')
plt.title("LSTM Glucose Level Prediction (First 200 Samples)")
plt.xlabel("Time Step")
plt.ylabel("Blood Glucose Level (mg/dL)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import joblib
# Save the final model and scaler in recommended formats
model.save('glucose_lstm_model.keras')  # Keras format
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(features, 'features.pkl')  # Save feature names
print("\nModel and scaler saved successfully!")